In [1]:
%load_ext autoreload
%autoreload 2

scene = "vienna_state_opera"
reconstruction_path = f"/home/mattia/Desktop/Repos/vggt/wrapper_output/{scene}/sparse" 
images_path = f"/home/mattia/Desktop/datasets/mydataset/data/{scene}/frames" 
depths_path = f"/home/mattia/Desktop/Repos/vggt/wrapper_output/{scene}/sparse/depth_maps"
sky_mask_path = f"/home/mattia/Desktop/datasets/mydataset/data/{scene}/depth_masks"
gt_path=f"/home/mattia/Desktop/datasets/mydataset/data/{scene}/colmap/sparse/0" 

# scene = "door"
# reconstruction_path = f"/home/mattia/Desktop/Repos/vggt/wrapper_output/{scene}/sparse" 
# images_path = f"/home/mattia/Desktop/Repos/wrapper_factory/benchmarks_3D/eth3d/{scene}/images_by_k" 
# depths_path = f"/home/mattia/Desktop/Repos/vggt/wrapper_output/{scene}/sparse/depth_maps"
# sky_mask_path = f"/home/mattia/Desktop/datasets/mydataset/data/{scene}/depth_masks"
# gt_path=f"/home/mattia/Desktop/Repos/wrapper_factory/benchmarks_3D/eth3d/{scene}/sparse" 

In [2]:
from adjuster_lm import AdjusterLM
adjuster = AdjusterLM(
    reconstruction_path = reconstruction_path,
    images_path = images_path,
    depths_path = depths_path,
    single_camera_per_folder=True,
    load_with_pad=False,
    grad_q=True,
    grad_t=True,
    grad_k=True,
    grad_z=False,
    detector_params={"low_threshold":0.20, "high_threshold":0.25, "kernel_size":7, "sigma":2},
)

adjuster.print_summary()

/home/mattia/miniconda3/envs/keypoint_factory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CannyEdgeDetector initialized with low_threshold=0.2, high_threshold=0.25, hysteresis=True, kernel_size=7, sigma=2, device=cuda
Found 106 images in /home/mattia/Desktop/datasets/mydataset/data/vienna_state_opera/frames


Extracting edges: 100%|██████████| 106/106 [00:00<00:00, 158.57it/s]


Max edges per image: 12,173


Computing viewgraph: 100%|██████████| 3971/3971 [00:10<00:00, 386.41it/s]


Filtered viewgraph: 2,853 pairs retained

Total parameters to optimize:
  Total:          750 parameters


                             Summary                              
------------------------------------------------------------------
Stage                           Time (s)         %    Per Iter
------------------------------------------------------------------
load_images                         1.46     8.2%            
load_poses_and_intrinsics           0.03     0.2%            
extract_edges                       0.67     3.8%            
load_depth_maps                     0.16     0.9%            
compute_distance_fields             4.90    27.6%            
compute_viewgraph                  10.42    58.8%            
------------------------------------------------------------------
Total                              17.73                       


In [3]:
import torch
from torch.nn import functional as F

def compute_batched_loss(residuals, robustifier=True, delta=1.0):
    """Vectorized batched loss computation."""

    # Group pairs: reshape (num_pairs*2,) -> (num_pairs, 2)
    residuals_pairs = residuals.view(-1, 2)  # (num_pairs, 2)

    # Sum i->j and j->i directions which are now in the same row
    if robustifier:
        # Use Huber loss for robust cost
        pair_losses = F.huber_loss(
            residuals_pairs,
            torch.zeros_like(residuals_pairs),
            reduction="none",
            delta=delta,
        )
    pair_losses = residuals_pairs.sum(dim=1)  # (num_pairs,)

    # Mean over pairs
    loss = pair_losses.mean()

    return loss

scene_model = adjuster.model
residuals = scene_model()
residuals.shape, compute_batched_loss(residuals)

(torch.Size([1710, 1]),
 tensor(5.4335, device='cuda:0', grad_fn=<MeanBackward0>))

In [4]:
from tqdm.auto import tqdm
import pypose as pp
from pypose.optim import LM
from pypose.optim.strategy import TrustRegion
from pypose.optim.scheduler import StopOnPlateau
from pypose.optim.kernel import Huber

# 1. Setup Kernel
kernel = Huber(delta=0.003)

# 2. Setup Strategy
# Note: 'radius' here is the INITIAL radius.
strategy = TrustRegion(radius=1e4, min=1e-8)

# 3. Setup Optimizer (Make sure to pass the strategy!)
optimizer = LM(scene_model, strategy=strategy, kernel=kernel, reject=16)

# 4. Scheduler
scheduler = StopOnPlateau(optimizer, steps=100, patience=101, decreasing=1e-4, verbose=False)

max_steps = 30
bar = tqdm(range(max_steps), desc="Optimizing")

for i in bar:
    # Perform one step
    loss = optimizer.step(input=None)
    
    # Update scheduler
    scheduler.step(loss)
    
    # Store loss
    scene_model.loss_list.append(loss.item())
    
    # --- DEBUGGING: Monitor the Radius ---
    # Access the dynamic radius using the path you found.
    # If this value drops rapidly (e.g., 1e4 -> 1e-1 -> 1e-5 ...), 
    # it confirms the optimizer is rejecting steps and getting stuck.
    # The optimizer updates 'pg' (parameter group) inside the strategy.update() method
    current_radius = optimizer.param_groups[0]['radius']
    
    bar.set_postfix({
        "loss": f"{loss.item():.4f}", 
        "radius": f"{current_radius:.2e}"
    })
    
    # # Optional: Break early if radius is too small (optimization effectively dead)
    # if current_radius < 1e-8:
    #     print(f"Stopping early at step {i}: Trust Region collapsed (radius < 1e-8).")
    #     # break
    
    # if scheduler.continual():
    #     print(f"Stopping early at step {i}: Scheduler triggered stop.")
    #     break

Optimizing:   0%|          | 0/30 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 13.49 GiB. GPU 

In [ ]:
from matplotlib import pyplot as plt
plt.plot(scene_model.loss_list)
plt.show()

In [ ]:
import os 
opt = f"optimized_reconstruction_LM/{scene}"
os.makedirs(opt, exist_ok=True)
scene_model.build_reconstruction_wrapper(opt, save_points=False)

In [ ]:
import sys
sys.path.append('/home/mattia/Desktop/Repos/wrapper_factory/benchmarks_3D')
from benchmark_pose import eval_colmap_model

# input = reconstruction_path
thresholds = [1,3,5]
print("AUC@",thresholds)
AUC_score_max, num_images, df_initial = eval_colmap_model(reconstruction_path, gt_path, return_df=True, thrs=thresholds)
print("VGGT AUC:   ",AUC_score_max)

try:
    ba = f"/home/mattia/Desktop/Repos/vggt/wrapper_output/{scene}/sparse_ba"
    # ba = f"/home/mattia/Desktop/Repos/wrapper_factory/benchmarks_3D/results/vggt_ba/synth/{scene}/sparse_ba"
    AUC_score_max, num_images, df_ba = eval_colmap_model(ba, gt_path, return_df=True, thrs=thresholds)
    print("VGGT+BA AUC:",AUC_score_max)
except:
    print("No BA reconstruction found.")

opt_GD = f"/home/mattia/Desktop/Repos/batchsfm/optimized_reconstruction_GD/{scene}"
AUC_score_max, num_images, df_optim = eval_colmap_model(opt_GD, gt_path, return_df=True, thrs=thresholds)
print("VGGT+EA (GD) AUC:",AUC_score_max)

AUC_score_max, num_images, df_optim = eval_colmap_model(opt, gt_path, return_df=True, thrs=thresholds)
print("VGGT+EA (LM) AUC:",AUC_score_max)